In [2]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# --- Langkah 1.1: Muat Data (Sudah Benar) ---
raw_data_path = '../dataset/dataset.csv'
try:
    df_raw = pd.read_csv(raw_data_path)
    df = df_raw.iloc[1:, 1:].reset_index(drop=True)
    df.columns = df_raw.columns[1:]
    print("Data berhasil dimuat (kolom 'Duration' sudah dihapus).")
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di path '{raw_data_path}'.")
    exit()

# --- Langkah 1.2: Buat Profil Pengguna (Tidak Berubah) ---
profile_col_names = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6']
user_profile_df = df[profile_col_names].copy()
user_profile_df.columns = ['age', 'gender', 'country', 'education', 'job_title', 'years_experience']
user_profile_df.insert(0, 'user_id', range(0, len(user_profile_df)))
print("DataFrame Profil Pengguna berhasil dibuat.")

# --- Langkah 1.3 (REVISI BESAR): Membuat Matriks Skill DAN Peta Kategori ---

# Peta dari awalan pertanyaan ke kategori yang kita definisikan
prefix_to_category = {
    'Q7': 'Bahasa Pemrograman',
    'Q8': 'Bahasa Rekomendasi',
    'Q9': 'Tools (IDE)',
    'Q10': 'Tools (Notebook)',
    'Q12': 'Hardware Khusus',
    'Q14': 'Library (Visualisasi)',
    'Q16': 'Library (Machine Learning)',
    'Q17': 'Algoritma (Machine Learning)',
    'Q27A': 'Platform (Cloud Computing)',
    'Q29A': 'Produk (Cloud Data)',
    'Q30A': 'Produk (Cloud Machine Learning)',
    'Q31A': 'Tools (Big Data)',
    'Q32A': 'Tools (Business Intelligence)',
    'Q36A': 'Platform (Berbagi Data)',
    'Q37A': 'Tools (AutoML)',
    'Q38A': 'Tools (Manajemen Eksperimen)',
    'Q40': 'Tools (Etika & Tata Kelola AI)'
}

skills_list = []
skill_category_map = {} # Kamus baru untuk menyimpan peta skill -> kategori

# Iterasi melalui peta kategori kita
for prefix, category in prefix_to_category.items():
    # Cari semua kolom yang cocok dengan awalan ini
    relevant_cols = [col for col in df.columns if col.startswith(prefix) and not col.endswith('OTHER')]
    
    for col_name in relevant_cols:
        skill_name = df[col_name].iloc[0]
        
        # Cek jika skill_name valid (bukan NaN atau string kosong)
        if pd.notna(skill_name) and skill_name.strip() != "":
            # Buat series biner
            skill_series = df[col_name].notna().astype(int).rename(skill_name)
            skills_list.append(skill_series)
            
            # Tambahkan ke peta kategori kita
            skill_category_map[skill_name] = category

# Gabungkan menjadi matriks User-Skill
user_skill_matrix = pd.concat(skills_list, axis=1)
user_skill_matrix = user_skill_matrix.loc[:, ~user_skill_matrix.columns.duplicated()]
user_skill_matrix.insert(0, 'user_id', range(0, len(user_skill_matrix)))
print("\nMatriks User-Skill berhasil dibuat.")

# Buat DataFrame dari peta kategori
skill_category_df = pd.DataFrame(list(skill_category_map.items()), columns=['Skill', 'Kategori'])
print("Peta Kategori Skill berhasil dibuat.")

# --- Langkah 1.4 (Dengan Penambahan): Menyimpan Semua Hasil Olahan ---
processed_profiles_path = '../dataset/processed_user_profiles.csv'
processed_skills_path = '../dataset/processed_user_skills.csv'
skill_categories_path = '../dataset/skill_categories.csv' # File baru!

try:
    user_profile_df.to_csv(processed_profiles_path, index=False)
    user_skill_matrix.to_csv(processed_skills_path, index=False)
    skill_category_df.to_csv(skill_categories_path, index=False)
    print(f"\nData olahan berhasil disimpan:")
    print(f"-> {processed_profiles_path}")
    print(f"-> {processed_skills_path}")
    print(f"-> {skill_categories_path}")
except Exception as e:
    print(f"\nTerjadi error saat menyimpan file: {e}")


Data berhasil dimuat (kolom 'Duration' sudah dihapus).
DataFrame Profil Pengguna berhasil dibuat.

Matriks User-Skill berhasil dibuat.
Peta Kategori Skill berhasil dibuat.

Data olahan berhasil disimpan:
-> ../dataset/processed_user_profiles.csv
-> ../dataset/processed_user_skills.csv
-> ../dataset/skill_categories.csv
